# Client SDK (040) Demo

Demonstrate the Python client SDK using a one-shot local socket server.


In [1]:
%load_ext autoreload
%autoreload 2
import json
import socket
import tempfile
import threading
from pathlib import Path

from ciphercache.client import Client, ClientConfig


In [ ]:
data_dir = Path(tempfile.mkdtemp(prefix="ciphercache-sdk-demo-"))
socket_path = data_dir / "ciphercached.sock"
tickets_dir = data_dir / "tickets"
tickets_dir.mkdir(parents=True, exist_ok=True)
(tickets_dir / "default.ticket").write_text("demo-token", encoding="utf-8")
data_dir


In [ ]:
def serve_once(response_envelope):
    if socket_path.exists():
        socket_path.unlink()
    def run():
        listener = socket.socket(socket.AF_UNIX, socket.SOCK_STREAM)
        try:
            listener.bind(str(socket_path))
            listener.listen(1)
            conn, _ = listener.accept()
            try:
                length_prefix = conn.recv(4)
                length = int.from_bytes(length_prefix, "big")
                _ = conn.recv(length)
                conn.sendall(encode_message(response_envelope))
            finally:
                conn.close()
        finally:
            listener.close()
            if socket_path.exists():
                socket_path.unlink()
    thread = threading.Thread(target=run, daemon=True)
    thread.start()
    return thread

client = Client(config=ClientConfig(data_dir=data_dir))

response = {
    "version": "v0",
    "id": "ping",
    "type": "response",
    "op": "ping",
    "payload": {"ok": True},
}
thread = serve_once(response)
client.ping()


In [ ]:
response = {
    "version": "v0",
    "id": "status",
    "type": "response",
    "op": "status",
    "payload": {"locked": False, "ttl_remaining_seconds": 120},
}
thread = serve_once(response)
client.status()


In [ ]:
response = {
    "version": "v0",
    "id": "secret",
    "type": "response",
    "op": "get_secret",
    "payload": {"secret": {"api_key": "demo"}},
}
thread = serve_once(response)
client.get_secret("service/api")
